## Cell 1 — Install the OpenAI Agents SDK

This notebook uses the `openai-agents` package. The cell below installs a pinned version so that every code example runs exactly as shown in the course.

> **To use the latest version instead**, run:
> ```
> pip install openai-agents
> ```
> Or substitute any version number you prefer after the `==`.

If the package is already installed at this version in your Colab session, this cell completes immediately.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.17.7 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.1/856.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00


## Cell 3 — Configure Your OpenAI API Key

This notebook uses **Google Colab Secrets** to retrieve your API key securely.

### Steps to add your key in Colab:
1. Click the **🔑 key icon** in the left sidebar (or go to **Runtime → Manage secrets**).
2. Click **+ Add new secret**.
3. Set **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the **Value**.
5. Enable the toggle to grant this notebook access.

> **Running locally?** Set the environment variable in your terminal before launching Jupyter:
> ```bash
> export OPENAI_API_KEY="sk-..."
> ```

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API key loaded ✓")

API key loaded ✓


## Cell 5 — Set the Model Name

All `Agent` definitions in this notebook use the `MODEL_NAME` variable declared below. Changing it here updates the model across the entire notebook.

See the [OpenAI models page](https://platform.openai.com/docs/models) for the latest available models.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 7 — Imports

Most of this lecture is about **inspecting JSON schemas**, not running full agent loops. The final demo cell adds `Agent`, `ModelSettings`, and `Runner`.

| Import | Purpose |
|---|---|
| `json` | Pretty-print generated schemas |
| `Enum` | Python enumeration type — maps to JSON Schema `"enum"` |
| `Annotated`, `Any`, `Literal` | Type annotation utilities |
| `TypedDict` | Typed dict structure supported as a tool parameter type |
| `BaseModel`, `Field` | Pydantic model and field constraints |
| `function_tool` | The decorator that generates a `FunctionTool` with JSON schema from a Python function |

In [4]:
import json
from enum import Enum
from typing import Annotated, Any, Literal

from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from agents import function_tool

## Cell 9 — How Schema Generation Works

This is a reference cell — no code to run. When you decorate a function with `@function_tool`, the SDK:

1. **Reads the docstring** to get the tool description and parameter descriptions.
2. **Inspects the type hints** on each parameter.
3. **Builds a Pydantic model** from those parameters.
4. **Generates a JSON schema** from that model.
5. **Applies strict-mode rules** if `strict_mode=True` (the default) — this is what we explore for the rest of the lecture.

The schema this process produces is the *only* thing the model ever sees about your tool. It never sees your Python source code.

## Cell 10 — Primitive Types: `str`, `int`, `float`, `bool`

Each Python primitive maps directly to a JSON Schema type:

| Python type | JSON Schema type |
|---|---|
| `str` | `"string"` |
| `int` | `"integer"` |
| `float` | `"number"` |
| `bool` | `"boolean"` |

In strict mode, all parameters appear in `"required"` and `"additionalProperties": false` is set at the top level.

In [5]:
@function_tool
def primitive_demo(
    name: str,
    age: int,
    score: float,
    is_active: bool,
) -> str:
    """Demonstrates primitive type annotation mapping."""
    return f"{name}, {age}, {score}, {is_active}"

print(json.dumps(primitive_demo.params_json_schema, indent=2))

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "age": {
      "title": "Age",
      "type": "integer"
    },
    "score": {
      "title": "Score",
      "type": "number"
    },
    "is_active": {
      "title": "Is Active",
      "type": "boolean"
    }
  },
  "required": [
    "name",
    "age",
    "score",
    "is_active"
  ],
  "title": "primitive_demo_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 12 — Strict Mode and Default Values

Here is the rule that catches almost every developer the first time: **in strict mode, every parameter is required, even ones with Python defaults.**

A parameter like `limit: int = 10` still has a perfectly normal Python default. But the JSON schema marks it required anyway, because strict mode does not distinguish "has a default" from "must be provided." The model must always send a value for every parameter.

Compare that to `strict_mode=False`, where parameters with defaults are genuinely optional in the schema, and the model can omit them.

In [6]:
# Non-strict mode: parameters with defaults are optional in the schema
@function_tool(strict_mode=False)
def optional_demo(
    query: str,
    limit: int = 10,
) -> str:
    """Demonstrates optional parameters with defaults in non-strict mode."""
    return f"query={query}, limit={limit}"

print("=== Non-strict mode (strict_mode=False) ===")
print(json.dumps(optional_demo.params_json_schema, indent=2))

print()

# Strict mode (default): ALL params required, even with Python defaults
@function_tool  # strict_mode=True is the default
def strict_optional_demo(
    query: str,
    limit: int = 10,
) -> str:
    """Strict mode: all params required even with defaults."""
    return f"query={query}, limit={limit}"

print("=== Strict mode (default) ===")
print(json.dumps(strict_optional_demo.params_json_schema, indent=2))

=== Non-strict mode (strict_mode=False) ===
{
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    },
    "limit": {
      "default": 10,
      "title": "Limit",
      "type": "integer"
    }
  },
  "required": [
    "query"
  ],
  "title": "optional_demo_args",
  "type": "object"
}

=== Strict mode (default) ===
{
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    },
    "limit": {
      "default": 10,
      "title": "Limit",
      "type": "integer"
    }
  },
  "required": [
    "query",
    "limit"
  ],
  "title": "strict_optional_demo_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 14 — Lists, and the Open-Dict Strict-Mode Trap

`list[T]` works fine in strict mode — it produces an `array` schema with an `items` sub-schema.

An open `dict[K, V]` parameter, like `dict[str, Any]`, does **not** work in strict mode. Strict mode requires `"additionalProperties": false` on every object, but a dict's whole purpose is to allow arbitrary keys. That conflict raises a `UserError` the moment the function is decorated.

**The fix:** use `strict_mode=False` if you genuinely need an open-ended dict, or — better — replace it with a `TypedDict` or `BaseModel` that names the fields explicitly, so you can stay in strict mode.

In [8]:
# list[str] — works fine in strict mode
@function_tool
def collection_demo(
    tags: list[str],
) -> str:
    """Demonstrates list type mapping."""
    return f"tags={tags}"

print("=== list[str] (strict mode) ===")
print(json.dumps(collection_demo.params_json_schema, indent=2))

print()

# dict[str, Any] — raises UserError in strict mode
# "@function_tool"  # ← This raises UserError
# def broken_dict_demo(metadata: dict[str, Any]) -> str: ...

# Fix option 1: drop to strict_mode=False
@function_tool(strict_mode=False)
def safe_dict_demo(metadata: dict[str, Any]) -> str:
    """Open-ended dict parameters require strict_mode=False."""
    return f"metadata={metadata}"

print("=== dict[str, Any] — requires strict_mode=False ===")
print(json.dumps(safe_dict_demo.params_json_schema, indent=2))

=== list[str] (strict mode) ===
{
  "properties": {
    "tags": {
      "items": {
        "type": "string"
      },
      "title": "Tags",
      "type": "array"
    }
  },
  "required": [
    "tags"
  ],
  "title": "collection_demo_args",
  "type": "object",
  "additionalProperties": false
}

=== dict[str, Any] — requires strict_mode=False ===
{
  "properties": {
    "metadata": {
      "additionalProperties": true,
      "title": "Metadata",
      "type": "object"
    }
  },
  "required": [
    "metadata"
  ],
  "title": "safe_dict_demo_args",
  "type": "object"
}


## Cell 16 — The Better Fix: Name the Fields Explicitly

Instead of reaching for `strict_mode=False`, ask whether the dict actually has a known shape. If it does, replace it with a `TypedDict` or `BaseModel`. The field names become part of the schema, and the tool stays in strict mode.

The cell below defines a `SearchFilters` `TypedDict` with two named fields instead of an open dict. The resulting nested schema gets `"additionalProperties": false` automatically, because a `TypedDict` has a closed, known set of keys.

In [9]:
class SearchFilters(TypedDict):
    category: str
    min_price: float

@function_tool  # strict_mode=True is the default — and it works here
def filtered_search(
    query: str,
    filters: SearchFilters,
) -> str:
    """Searches using a strictly typed filters object instead of an open dict."""
    return f"query={query}, filters={filters}"

print("=== TypedDict instead of dict[K, V] — stays in strict mode ===")
print(json.dumps(filtered_search.params_json_schema, indent=2))

=== TypedDict instead of dict[K, V] — stays in strict mode ===
{
  "$defs": {
    "SearchFilters": {
      "properties": {
        "category": {
          "title": "Category",
          "type": "string"
        },
        "min_price": {
          "title": "Min Price",
          "type": "number"
        }
      },
      "required": [
        "category",
        "min_price"
      ],
      "title": "SearchFilters",
      "type": "object",
      "additionalProperties": false
    }
  },
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    },
    "filters": {
      "$ref": "#/$defs/SearchFilters"
    }
  },
  "required": [
    "query",
    "filters"
  ],
  "title": "filtered_search_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 18 — `Literal` Types and Python `Enum`

Both constrain a parameter to a fixed set of values. The SDK maps both to an `"enum"` array in the JSON schema. Both are fully supported in strict mode.

| Approach | Best used when |
|---|---|
| `Literal["low", "medium", "high"]` | One-off, inline use in a single tool |
| Python `Enum` class | The same set of values appears in multiple tools |

In [10]:
class Priority(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

@function_tool
def priority_demo(
    task: str,
    priority: Literal["low", "medium", "high"],
) -> str:
    """Demonstrates Literal type mapping."""
    return f"task={task}, priority={priority}"

@function_tool
def enum_demo(
    task: str,
    priority: Priority,
) -> str:
    """Demonstrates Enum type mapping."""
    return f"task={task}, priority={priority}"

print("=== Literal type ===")
print(json.dumps(priority_demo.params_json_schema, indent=2))

print()
print("=== Python Enum ===")
print(json.dumps(enum_demo.params_json_schema, indent=2))

=== Literal type ===
{
  "properties": {
    "task": {
      "title": "Task",
      "type": "string"
    },
    "priority": {
      "enum": [
        "low",
        "medium",
        "high"
      ],
      "title": "Priority",
      "type": "string"
    }
  },
  "required": [
    "task",
    "priority"
  ],
  "title": "priority_demo_args",
  "type": "object",
  "additionalProperties": false
}

=== Python Enum ===
{
  "$defs": {
    "Priority": {
      "enum": [
        "low",
        "medium",
        "high"
      ],
      "title": "Priority",
      "type": "string"
    }
  },
  "properties": {
    "task": {
      "title": "Task",
      "type": "string"
    },
    "priority": {
      "$ref": "#/$defs/Priority"
    }
  },
  "required": [
    "task",
    "priority"
  ],
  "title": "enum_demo_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 20 — Optional Types — `str | None`

`str | None` (equivalent to `Optional[str]`) produces `anyOf` with two variants: `{type: string}` and `{type: null}`. This works cleanly in strict mode — the field is still pushed into `"required"`, and the model sends the JSON literal `null` when it has nothing meaningful to provide.

In [11]:
@function_tool  # strict_mode=True is the default
def union_demo(
    city: str,
    country: str | None = None,
) -> str:
    """Demonstrates Optional type mapping under strict mode."""
    return f"city={city}, country={country}"

print("=== str | None under strict mode ===")
print(json.dumps(union_demo.params_json_schema, indent=2))

=== str | None under strict mode ===
{
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    },
    "country": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "title": "Country"
    }
  },
  "required": [
    "city",
    "country"
  ],
  "title": "union_demo_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 22 — Nested Pydantic Models as Parameters

`BaseModel` subclasses are the recommended pattern for complex tool inputs.

- The SDK recursively generates the JSON schema for the nested model.
- In strict mode, every field at every nesting level is required, and `additionalProperties: false` is set throughout.
- Pydantic validates the model's JSON input before your function is called.

In [12]:
class Address(BaseModel):
    street: str
    city: str
    postcode: str

@function_tool
def nested_demo(
    name: str,
    address: Address,
) -> str:
    """Demonstrates nested Pydantic model as a parameter."""
    return f"{name} lives at {address.city}"

print("=== Nested Pydantic model (strict mode) ===")
print(json.dumps(nested_demo.params_json_schema, indent=2))

=== Nested Pydantic model (strict mode) ===
{
  "$defs": {
    "Address": {
      "properties": {
        "street": {
          "title": "Street",
          "type": "string"
        },
        "city": {
          "title": "City",
          "type": "string"
        },
        "postcode": {
          "title": "Postcode",
          "type": "string"
        }
      },
      "required": [
        "street",
        "city",
        "postcode"
      ],
      "title": "Address",
      "type": "object",
      "additionalProperties": false
    }
  },
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "address": {
      "$ref": "#/$defs/Address"
    }
  },
  "required": [
    "name",
    "address"
  ],
  "title": "nested_demo_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 24 — `Annotated` with `Field` Constraints

`Annotated[T, Field(...)]` attaches Pydantic constraints directly to a type annotation. These appear in the generated JSON schema and are enforced by Pydantic before your function is called.

| `Field` argument | JSON Schema key | Effect |
|---|---|---|
| `ge=0, le=100` | `"minimum"`, `"maximum"` | Integer or float range |
| `min_length=3, max_length=20` | `"minLength"`, `"maxLength"` | String length bounds |
| `description="..."` | `"description"` | Human-readable field description |

In [13]:
@function_tool
def constrained_demo(
    score: Annotated[int, Field(ge=0, le=100, description="Score between 0 and 100")],
    username: Annotated[str, Field(min_length=3, max_length=20, description="Username (3 to 20 characters)")],
) -> str:
    """Demonstrates Field constraints in schema generation."""
    return f"score={score}, user={username}"

print("=== Annotated[T, Field(...)] constraints ===")
print(json.dumps(constrained_demo.params_json_schema, indent=2))

=== Annotated[T, Field(...)] constraints ===
{
  "properties": {
    "score": {
      "description": "Score between 0 and 100",
      "maximum": 100,
      "minimum": 0,
      "title": "Score",
      "type": "integer"
    },
    "username": {
      "description": "Username (3 to 20 characters)",
      "maxLength": 20,
      "minLength": 3,
      "title": "Username",
      "type": "string"
    }
  },
  "required": [
    "score",
    "username"
  ],
  "title": "constrained_demo_args",
  "type": "object",
  "additionalProperties": false
}


## Cell 26 — Strict Mode Rules — Reference Table

Use this table when deciding whether a tool needs `strict_mode=False`.

| Pattern | Strict mode OK? | Notes |
|---|---|---|
| `str`, `int`, `float`, `bool` | Yes | Direct mapping to JSON Schema primitives |
| Parameters with Python defaults | Required anyway | Strict mode requires every parameter, regardless of defaults |
| `list[str]`, `list[int]`, `list[BaseModel]` | Yes | Items schema generated recursively |
| `dict[str, Any]` (open dict) | No | Raises `UserError`. Use `strict_mode=False`, or replace with `TypedDict`/`BaseModel` |
| `Optional[str]` / `str \| None` | Yes | Produces `anyOf`, fully supported |
| Nested `BaseModel` | Yes | Recursive strict schema |
| `TypedDict` | Yes | Same schema as `BaseModel` |
| `Literal[...]` / `Enum` | Yes | Produces `"enum"` constraint |
| `Annotated[T, Field(...)]` | Yes | Constraints included in schema |

## Cell 27 — Live Demo: Agent with a Nested Pydantic Tool

This cell runs a real agent. A `TaskInput` Pydantic model captures `title`, `priority` (a `Literal` set), and `due_days`. A `create_task` tool accepts `TaskInput` as its single parameter.

**What to observe:** the model generates a complete JSON object for `TaskInput` in one call. The SDK validates it against the Pydantic model before calling `create_task`.

In [14]:
from agents import Agent, ModelSettings, Runner

class TaskInput(BaseModel):
    title: str
    priority: Literal["low", "medium", "high"]
    due_days: int

@function_tool
def create_task(task: TaskInput) -> str:
    """Creates a new task with the given details.

    Args:
        task: The task details including title, priority and due days.
    """
    return (
        f"Task created: '{task.title}' "
        f"with {task.priority} priority, "
        f"due in {task.due_days} days."
    )

agent = Agent(
    name="Task Manager",
    instructions=(
        "You are a task management assistant. "
        "Use the create_task tool to create tasks from user requests."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(temperature=0),
    tools=[create_task],
)

result = await Runner.run(
    agent,
    "Create a high priority task to review the Q4 report, due in 3 days.",
)
print(result.final_output)

Done — I created the task: “Review the Q4 report” with high priority, due in 3 days.
